In [1]:
        import torch
import torchvision
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, Subset
import timm
from torch import nn

In [2]:
def setup_model():
    model = timm.create_model('inception_v4', pretrained=True)
    model.eval()
    return model

In [3]:
pretrained_model = setup_model()

In [4]:
transform = transforms.Compose([
    transforms.ToTensor()
])
cifar10_dataset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
cat_indices = [i for i, label in enumerate(cifar10_dataset.targets) if label == 3]
# Create a subset of the CIFAR-10 dataset containing only cat images
cat_subset = Subset(cifar10_dataset, cat_indices)
data_loader = DataLoader(cat_subset, batch_size=32, shuffle=True)

Files already downloaded and verified


In [5]:
import torch
import torch.nn as nn

class MyModel(nn.Module):
    def __init__(self, pretrained_model):
        super(MyModel, self).__init__()
        
        self.upsample = nn.Upsample(size=(299, 299), mode='bilinear', align_corners=False)
        self.pretrained_model = pretrained_model
        self.pretrained_model.reset_classifier(0)  # 0 means no output classes
        self.pretrained_model.eval()

    def forward(self, x):
        x = self.upsample(x)
        x = self.pretrained_model(x)
        return x


In [6]:
model = MyModel(pretrained_model=pretrained_model)

In [8]:
from src.vmmd import VMMD

vmmd = VMMD(batch_size=100)
vmmd.fit(cat_subset, embedding_function=model)

Epoch 0 of 30


KeyboardInterrupt: 